# Customer Support Agent - Demo Notebook

This notebook brings together all the components of the `customer_support_agent` package in a logical sequence:

1. **Settings & Configuration**
2. **Database Initialization & Repositories**
3. **Knowledge Base (RAG) Ingestion & Search**
4. **Customer Memory Store (Mem0)**
5. **Support Tools**
6. **Copilot Agent — Draft Generation**
7. **Draft Service — End-to-End Workflow**

## 0. Imports & Path Setup

In [1]:
import re
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/

# Pattern to match any variable/key that looks like an API key reference
api_key_pattern = re.compile(r'[a-zA-Z_]*api[_]?key', re.IGNORECASE)

# File extensions to scan
extensions = {'.py', '.env', '.yaml', '.yml', '.toml', '.json', '.md', '.cfg', '.ini', '.ipynb'}

results = {}
for file_path in sorted(PROJECT_ROOT.rglob('*')):
    if not file_path.is_file():
        continue
    if file_path.suffix not in extensions:
        continue
    # Skip hidden dirs / __pycache__ / .git / data dirs
    parts = file_path.relative_to(PROJECT_ROOT).parts
    if any(p.startswith('.') or p == '__pycache__' or p == 'data' for p in parts):
        continue
    try:
        text = file_path.read_text(encoding='utf-8', errors='ignore')
    except Exception:
        continue
    matches = set()
    for line_no, line in enumerate(text.splitlines(), 1):
        for m in api_key_pattern.finditer(line):
            matches.add((line_no, m.group()))
    if matches:
        rel = file_path.relative_to(PROJECT_ROOT)
        results[str(rel)] = sorted(matches)

# Print report
print(f"Found api_key references in {len(results)} file(s):\n")
for file, hits in results.items():
    print(f"📄 {file}")
    for line_no, match in hits:
        print(f"     Line {line_no:>4d} : {match}")
    print()

Found api_key references in 9 file(s):

📄 customer_support_agent\core\settings.py
     Line   19 : groq_api_key
     Line   24 : openai_api_key
     Line   26 : google_api_key

📄 customer_support_agent\integrations\memory\mem0_store.py
     Line   24 : api_key
     Line   24 : openai_api_key
     Line   36 : google_api_key
     Line   40 : api_key
     Line   40 : google_api_key
     Line   45 : openai_api_key
     Line   49 : api_key
     Line   49 : openai_api_key
     Line   63 : GOOGLE_API_KEY
     Line   64 : OPENAI_API_KEY

📄 customer_support_agent\integrations\rag\chroma_kb.py
     Line   19 : google_api_key
     Line   31 : google_api_key
     Line   32 : GOOGLE_API_KEY
     Line   33 : GOOGLE_API_KEY
     Line   33 : google_api_key
     Line   40 : GOOGLE_API_KEY

📄 customer_support_agent\services\copilot_service.py
     Line   23 : openai_api_key
     Line   25 : GROQ_API_KEY
     Line   30 : groq_api_key
     Line   35 : api_key
     Line   35 : openai_api_key

📄 docs\EC2_de

In [ ]:
import sys, os, json, re, hashlib, logging, sqlite3
from pathlib import Path
from functools import lru_cache
from typing import Any, Callable

# Ensure the project root is the working directory
PROJECT_ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: c:\ML\AgenticAI\customer_support_agent


## 1. Settings & Configuration

All configuration lives in `customer_support_agent.core.settings`.  
It reads from a `.env` file and exposes paths, API keys, model names, etc.

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_file_encoding="utf-8",
        extra="ignore",
    )

    app_name: str = "AI Copilot for Support Agents"

    groq_api_key: str = ""
    groq_model: str = ""
    llm_temperature: float = 0.2

    openai_api_key: str = ""
    openai_model: str = "gpt-4o"
    google_api_key: str = ""
    google_embedding_model: str = "gemini-embedding-001"
    enable_local_embeddings: bool = False

    workspace_dir: Path = Path.cwd()
    data_dir: Path = Path("data")
    db_path: Path = Path("data/support.db")
    chroma_rag_dir: Path = Path("data/chroma_rag")
    chroma_mem0_dir: Path = Path("data/chroma_mem0")
    knowledge_base_dir: Path = Path("knowledge_base")

    rag_chunk_size: int = 800
    rag_chunk_overlap: int = 120
    rag_top_k: int = 4
    mem0_top_k: int = 5

    api_host: str = "0.0.0.0"
    api_port: int = 8000
    dashboard_api_url: str = "http://localhost:8000"

    def resolve(self, path: Path) -> Path:
        return path if path.is_absolute() else self.workspace_dir / path

    @property
    def db_file(self) -> Path:
        return self.resolve(self.db_path)

    @property
    def chroma_rag_path(self) -> Path:
        return self.resolve(self.chroma_rag_dir)

    @property
    def chroma_mem0_path(self) -> Path:
        return self.resolve(self.chroma_mem0_dir)

    @property
    def knowledge_base_path(self) -> Path:
        return self.resolve(self.knowledge_base_dir)

    @property
    def effective_google_embedding_model(self) -> str:
        model = (self.google_embedding_model or "").strip()
        if not model:
            return "gemini-embedding-001"
        if model.startswith("models/"):
            model = model[len("models/"):]
        deprecated_aliases = {
            "text-embedding-004", "embedding-001", "embedding-gecko-001",
            "gemini-embedding-exp", "gemini-embedding-exp-03-07",
        }
        if model in deprecated_aliases:
            return "gemini-embedding-001"
        return model


@lru_cache
def get_settings() -> Settings:
    return Settings()


def ensure_directories(settings: Settings | None = None) -> None:
    config = settings or get_settings()
    for path in (
        config.resolve(config.data_dir),
        config.chroma_rag_path,
        config.chroma_mem0_path,
        config.knowledge_base_path,
    ):
        path.mkdir(parents=True, exist_ok=True)


settings = get_settings()
ensure_directories(settings)

print(f"App name          : {settings.app_name}")
print(f"OpenAI model      : {settings.openai_model}")
print(f"Embedding model   : {settings.effective_google_embedding_model}")
print(f"DB path           : {settings.db_file}")
print(f"Chroma RAG path   : {settings.chroma_rag_path}")
print(f"Chroma Mem0 path  : {settings.chroma_mem0_path}")
print(f"Knowledge base    : {settings.knowledge_base_path}")
print(f"RAG chunk size    : {settings.rag_chunk_size}")
print(f"RAG top_k         : {settings.rag_top_k}")
print(f"Mem0 top_k        : {settings.mem0_top_k}")

App name          : AI Copilot for Support Agents
OpenAI model      : gpt-4o
Embedding model   : gemini-embedding-001
DB path           : C:\ML\AgenticAI\customer_support_agent\data\support.db
Chroma RAG path   : C:\ML\AgenticAI\customer_support_agent\data\chroma_rag
Chroma Mem0 path  : C:\ML\AgenticAI\customer_support_agent\data\chroma_mem0
Knowledge base    : C:\ML\AgenticAI\customer_support_agent\knowledge_base
RAG chunk size    : 800
RAG top_k         : 4
Mem0 top_k        : 5


## 2. Database Initialization & Repositories

SQLite tables for **customers**, **tickets**, and **drafts** are created by `init_db()`.

In [ ]:
# ── Database helpers ──────────────────────────────────────────────────────────

def connect() -> sqlite3.Connection:
    ensure_directories(settings)
    conn = sqlite3.connect(str(settings.db_file), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def row_to_dict(row: sqlite3.Row | None) -> dict[str, Any] | None:
    if row is None:
        return None
    return dict(row)


def init_db() -> None:
    with connect() as conn:
        conn.executescript("""
            CREATE TABLE IF NOT EXISTS customers (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT UNIQUE NOT NULL,
                name TEXT,
                company TEXT,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            CREATE TABLE IF NOT EXISTS tickets (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                customer_id INTEGER REFERENCES customers(id),
                subject TEXT NOT NULL,
                description TEXT NOT NULL,
                status TEXT DEFAULT 'open',
                priority TEXT DEFAULT 'medium',
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            CREATE TABLE IF NOT EXISTS drafts (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                ticket_id INTEGER REFERENCES tickets(id),
                content TEXT NOT NULL,
                context_used TEXT,
                status TEXT DEFAULT 'pending',
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
            CREATE TRIGGER IF NOT EXISTS tickets_updated_at_trigger
            AFTER UPDATE ON tickets
            FOR EACH ROW
            BEGIN
                UPDATE tickets SET updated_at = CURRENT_TIMESTAMP WHERE id = OLD.id;
            END;
        """)


# ── Repository classes ────────────────────────────────────────────────────────

class CustomersRepository:
    def create_or_get(self, email: str, name: str | None = None, company: str | None = None) -> dict[str, Any]:
        with connect() as conn:
            row = conn.execute("SELECT * FROM customers WHERE email = ?", (email,)).fetchone()
            if row:
                updates, values = [], []
                if name and not row["name"]:
                    updates.append("name = ?"); values.append(name)
                if company and not row["company"]:
                    updates.append("company = ?"); values.append(company)
                if updates:
                    values.append(email)
                    conn.execute(f"UPDATE customers SET {', '.join(updates)} WHERE email = ?", values)
                refreshed = conn.execute("SELECT * FROM customers WHERE email = ?", (email,)).fetchone()
                return row_to_dict(refreshed) or {}
            conn.execute("INSERT INTO customers (email, name, company) VALUES (?, ?, ?)", (email, name, company))
            created = conn.execute("SELECT * FROM customers WHERE email = ?", (email,)).fetchone()
            return row_to_dict(created) or {}

    def get_by_id(self, customer_id: int) -> dict[str, Any] | None:
        with connect() as conn:
            row = conn.execute("SELECT * FROM customers WHERE id = ?", (customer_id,)).fetchone()
            return row_to_dict(row)

    def get_by_email(self, email: str) -> dict[str, Any] | None:
        with connect() as conn:
            row = conn.execute("SELECT * FROM customers WHERE email = ?", (email,)).fetchone()
            return row_to_dict(row)


class TicketsRepository:
    def create(self, customer_id: int, subject: str, description: str, priority: str = "medium", status: str = "open") -> dict[str, Any]:
        with connect() as conn:
            cursor = conn.execute(
                "INSERT INTO tickets (customer_id, subject, description, priority, status) VALUES (?, ?, ?, ?, ?)",
                (customer_id, subject, description, priority, status),
            )
            row = conn.execute("SELECT * FROM tickets WHERE id = ?", (cursor.lastrowid,)).fetchone()
            return row_to_dict(row) or {}

    def list(self, limit: int = 100) -> list[dict[str, Any]]:
        with connect() as conn:
            rows = conn.execute(
                "SELECT t.*, c.email AS customer_email, c.name AS customer_name, c.company AS customer_company "
                "FROM tickets t JOIN customers c ON c.id = t.customer_id ORDER BY t.created_at DESC LIMIT ?",
                (limit,),
            ).fetchall()
            return [dict(row) for row in rows]

    def get_by_id(self, ticket_id: int) -> dict[str, Any] | None:
        with connect() as conn:
            row = conn.execute(
                "SELECT t.*, c.email AS customer_email, c.name AS customer_name, c.company AS customer_company "
                "FROM tickets t JOIN customers c ON c.id = t.customer_id WHERE t.id = ?",
                (ticket_id,),
            ).fetchone()
            return row_to_dict(row)

    def set_status(self, ticket_id: int, status: str) -> dict[str, Any] | None:
        with connect() as conn:
            conn.execute("UPDATE tickets SET status = ? WHERE id = ?", (status, ticket_id))
            row = conn.execute("SELECT * FROM tickets WHERE id = ?", (ticket_id,)).fetchone()
            return row_to_dict(row)

    def count_open_for_customer(self, customer_email: str) -> int:
        with connect() as conn:
            row = conn.execute(
                "SELECT COUNT(*) AS open_count FROM tickets t JOIN customers c ON c.id = t.customer_id "
                "WHERE c.email = ? AND t.status = 'open'",
                (customer_email,),
            ).fetchone()
            return int(row["open_count"]) if row else 0


class DraftsRepository:
    def create(self, ticket_id: int, content: str, context_used: str | None = None, status: str = "pending") -> dict[str, Any]:
        with connect() as conn:
            cursor = conn.execute(
                "INSERT INTO drafts (ticket_id, content, context_used, status) VALUES (?, ?, ?, ?)",
                (ticket_id, content, context_used, status),
            )
            row = conn.execute("SELECT * FROM drafts WHERE id = ?", (cursor.lastrowid,)).fetchone()
            return row_to_dict(row) or {}

    def get_latest_for_ticket(self, ticket_id: int) -> dict[str, Any] | None:
        with connect() as conn:
            row = conn.execute(
                "SELECT * FROM drafts WHERE ticket_id = ? ORDER BY created_at DESC LIMIT 1",
                (ticket_id,),
            ).fetchone()
            return row_to_dict(row)

    def get_by_id(self, draft_id: int) -> dict[str, Any] | None:
        with connect() as conn:
            row = conn.execute("SELECT * FROM drafts WHERE id = ?", (draft_id,)).fetchone()
            return row_to_dict(row)

    def update(self, draft_id: int, content: str | None = None, status: str | None = None) -> dict[str, Any] | None:
        updates, values = [], []
        if content is not None:
            updates.append("content = ?"); values.append(content)
        if status is not None:
            updates.append("status = ?"); values.append(status)
        if not updates:
            return self.get_by_id(draft_id)
        with connect() as conn:
            values.append(draft_id)
            conn.execute(f"UPDATE drafts SET {', '.join(updates)} WHERE id = ?", values)
            row = conn.execute("SELECT * FROM drafts WHERE id = ?", (draft_id,)).fetchone()
            return row_to_dict(row)

    def get_ticket_and_customer_by_draft(self, draft_id: int) -> dict[str, Any] | None:
        with connect() as conn:
            row = conn.execute(
                "SELECT d.id AS draft_id, d.ticket_id, d.content AS draft_content, d.status AS draft_status, "
                "t.subject, t.description, t.status AS ticket_status, "
                "c.id AS customer_id, c.email AS customer_email, c.name AS customer_name, c.company AS customer_company "
                "FROM drafts d JOIN tickets t ON t.id = d.ticket_id JOIN customers c ON c.id = t.customer_id WHERE d.id = ?",
                (draft_id,),
            ).fetchone()
            return row_to_dict(row)


# Create tables and instantiate repos
init_db()
print("Database initialized.")

customers_repo = CustomersRepository()
tickets_repo = TicketsRepository()
drafts_repo = DraftsRepository()

Database initialized.


### 2a. Create or get a customer

In [4]:
customer = customers_repo.create_or_get(
    email="alex@acme.io",
    name="Alex Rivera",
    company="Acme Labs",
)
print("Customer:", json.dumps(customer, indent=2, default=str))

Customer: {
  "id": 1,
  "email": "alex@acme.io",
  "name": "Alex Rivera",
  "company": "Acme Labs",
  "created_at": "2026-03-29 07:17:29"
}


### 2b. Create a support ticket

In [5]:
ticket = tickets_repo.create(
    customer_id=customer["id"],
    subject="Unable to withdraw cash from ATM",
    description=(
        "I tried withdrawing INR 10,000 from the ATM near MG Road branch but the "
        "transaction failed twice. My account was debited but I did not receive the cash. "
        "Please reverse the amount urgently."
    ),
    priority="high",
)
print("Ticket:", json.dumps(ticket, indent=2, default=str))

Ticket: {
  "id": 5,
  "customer_id": 1,
  "subject": "Unable to withdraw cash from ATM",
  "description": "I tried withdrawing INR 10,000 from the ATM near MG Road branch but the transaction failed twice. My account was debited but I did not receive the cash. Please reverse the amount urgently.",
  "status": "open",
  "priority": "high",
  "created_at": "2026-03-29 07:40:17",
  "updated_at": "2026-03-29 07:40:17"
}


### 2c. List all tickets

In [6]:
all_tickets = tickets_repo.list(limit=5)
for t in all_tickets:
    print(f"  #{t['id']}  {t['status']:6s}  {t['priority']:6s}  {t['customer_email']}  {t['subject']}")

  #5  open    high    alex@acme.io  Unable to withdraw cash from ATM
  #4  open    high    alex@acme.io  Unable to withdraw cash from ATM
  #3  open    high    alex@acme.io  Unable to withdraw cash from ATM
  #2  open    high    alex@acme.io  Unable to withdraw cash from ATM
  #1  open    high    alex@acme.io  Unable to withdraw cash from ATM


## 3. Knowledge Base — Ingestion & Search

`KnowledgeBaseService` chunks markdown files from `knowledge_base/` and stores them in ChromaDB with Gemini embeddings.

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter


class KnowledgeBaseService:
    def __init__(self, settings: Settings):
        self._settings = settings
        self._client = chromadb.PersistentClient(path=str(settings.chroma_rag_path))
        self._collection_name = "support_kb_gemini" if settings.google_api_key else "support_kb"
        self._embedding_function = self._build_embedding_function()
        self._collection = self._client.get_or_create_collection(
            name=self._collection_name,
            embedding_function=self._embedding_function,
        )
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.rag_chunk_size,
            chunk_overlap=settings.rag_chunk_overlap,
        )

    def _build_embedding_function(self) -> Any:
        if self._settings.google_api_key:
            os.environ.setdefault("GOOGLE_API_KEY", self._settings.google_api_key)
            try:
                return embedding_functions.GoogleGenaiEmbeddingFunction(
                    model_name=self._settings.effective_google_embedding_model,
                )
            except Exception as exc:
                raise RuntimeError(
                    "Gemini embedding initialization failed. Install `google-genai` and verify GOOGLE_API_KEY."
                ) from exc
        return embedding_functions.DefaultEmbeddingFunction()

    def ingest_directory(self, directory: Path, clear_existing: bool = False) -> dict[str, int]:
        if clear_existing:
            self._client.delete_collection(name=self._collection_name)
            self._collection = self._client.get_or_create_collection(
                name=self._collection_name,
                embedding_function=self._embedding_function,
            )
        source_files = sorted([*directory.glob("*.md"), *directory.glob("*.txt")])
        docs, ids, metadatas = [], [], []
        for file_path in source_files:
            text = file_path.read_text(encoding="utf-8")
            chunks = self._splitter.split_text(text)
            for index, chunk in enumerate(chunks):
                chunk_hash = hashlib.sha1(chunk.encode("utf-8")).hexdigest()[:10]
                doc_id = f"{file_path.stem}-{index}-{chunk_hash}"
                docs.append(chunk)
                ids.append(doc_id)
                metadatas.append({"source": file_path.name, "chunk_index": index})
        if docs:
            self._collection.upsert(documents=docs, ids=ids, metadatas=metadatas)
        return {
            "files_indexed": len(source_files),
            "chunks_indexed": len(docs),
            "collection_count": self._collection.count(),
        }

    def search(self, query: str, top_k: int | None = None) -> list[dict[str, Any]]:
        if self._collection.count() == 0:
            return []
        results = self._collection.query(
            query_texts=[query],
            n_results=top_k or self._settings.rag_top_k,
            include=["documents", "metadatas", "distances"],
        )
        documents = (results.get("documents") or [[]])[0]
        metadatas = (results.get("metadatas") or [[]])[0]
        distances = (results.get("distances") or [[]])[0]
        combined = []
        for i, document in enumerate(documents):
            metadata = metadatas[i] if i < len(metadatas) else {}
            distance = distances[i] if i < len(distances) else None
            combined.append({"content": document, "source": metadata.get("source", "unknown"), "distance": distance})
        return combined


kb = KnowledgeBaseService(settings=settings)

# Ingest the knowledge base directory
ingest_result = kb.ingest_directory(
    directory=settings.knowledge_base_path,
    clear_existing=False,
)
print("Ingest result:", json.dumps(ingest_result, indent=2))

Ingest result: {
  "files_indexed": 4,
  "chunks_indexed": 4,
  "collection_count": 4
}


In [8]:
# Search the knowledge base
kb_results = kb.search("ATM cash withdrawal failed but account debited", top_k=3)
for i, hit in enumerate(kb_results, 1):
    print(f"\n--- KB Hit {i} (source: {hit['source']}, distance: {hit['distance']:.4f}) ---")
    print(hit["content"][:300])


--- KB Hit 1 (source: banking-atm-cash-withdrawal-faq.md, distance: 0.2745) ---
# ATM Cash Withdrawal FAQ

## Daily Withdrawal Limits

- Standard savings account: up to INR 25,000 per day.
- Premium savings account: up to INR 50,000 per day.
- Limits may vary by card type and account risk profile.

## Common Issues

### Cash debited but not dispensed

- Reversal is typically pr

--- KB Hit 2 (source: banking-charges-and-minimum-balance.md, distance: 0.4809) ---
# Banking Charges and Minimum Balance

## Minimum Balance Rules

- Urban branch savings account: minimum monthly average balance INR 10,000.
- Semi-urban branch savings account: minimum monthly average balance INR 5,000.
- Rural branch savings account: minimum monthly average balance INR 2,500.

## 

--- KB Hit 3 (source: saving-account-rule.md, distance: 0.5005) ---
# Saving Account Rule

This document defines key policy rules for saving account holders.

## Mandatory Rules

1. Cheque services are activated only after 1 month 

## 4. Customer Memory Store (Mem0)

`CustomerMemoryStore` wraps Mem0 to store and recall per-customer interaction history.

In [ ]:
from mem0 import Memory


class CustomerMemoryStore:
    def __init__(self, settings: Settings, llm: Any):
        _ = llm
        config: dict[str, Any] = {
            "llm": {
                "provider": "openai",
                "config": {
                    "model": settings.openai_model,
                    "api_key": settings.openai_api_key,
                    "temperature": settings.llm_temperature,
                },
            },
            "vector_store": {
                "provider": "chroma",
                "config": {
                    "path": str(settings.chroma_mem0_path),
                },
            },
        }
        if settings.google_api_key:
            config["embedder"] = {
                "provider": "gemini",
                "config": {
                    "api_key": settings.google_api_key,
                    "model": settings.effective_google_embedding_model,
                },
            }
        elif settings.openai_api_key:
            config["embedder"] = {
                "provider": "openai",
                "config": {"api_key": settings.openai_api_key},
            }
        elif settings.enable_local_embeddings:
            config["embedder"] = {
                "provider": "huggingface",
                "config": {"model": "all-MiniLM-L6-v2"},
            }
        else:
            raise RuntimeError(
                "No embedding provider configured for Mem0. Set GOOGLE_API_KEY or OPENAI_API_KEY."
            )
        self._memory = Memory.from_config(config)

    def search(self, query: str, user_id: str, limit: int = 5) -> list[dict[str, Any]]:
        try:
            raw = self._memory.search(query, user_id=user_id, limit=limit)
        except TypeError:
            raw = self._memory.search(query, user_id=user_id)
        return self._normalize_results(raw, limit)

    def list_memories(self, user_id: str, limit: int = 20) -> list[dict[str, Any]]:
        if hasattr(self._memory, "get_all"):
            raw = self._memory.get_all(user_id=user_id)
            return self._normalize_results(raw, limit)

    def add_interaction(self, user_id: str, user_input: str, assistant_response: str, metadata: dict[str, Any] | None = None) -> None:
        messages = [
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": assistant_response},
        ]
        self._add_messages(messages=messages, user_id=user_id, metadata=metadata)

    def add_resolution(self, user_id: str, ticket_subject: str, ticket_description: str, accepted_draft: str, entity_links: list[str] | None = None) -> None:
        entity_text = ""
        if entity_links:
            entity_text = "\nLinked entities: " + ", ".join(entity_links)
        messages = [
            {"role": "user", "content": f"Ticket subject: {ticket_subject}\nProblem: {ticket_description}"},
            {"role": "assistant", "content": f"Resolution accepted by support agent:\n{accepted_draft}{entity_text}"},
        ]
        self._add_messages(messages=messages, user_id=user_id, metadata={"type": "resolution"})

    def _add_messages(self, messages: list[dict[str, str]], user_id: str, metadata: dict[str, Any] | None = None) -> None:
        try:
            self._memory.add(messages, user_id=user_id, metadata=metadata or {})
        except TypeError:
            self._memory.add(messages, user_id=user_id)

    def _normalize_results(self, raw: Any, limit: int) -> list[dict[str, Any]]:
        items = []
        if isinstance(raw, dict) and "results" in raw:
            iterable = raw.get("results") or []
        elif isinstance(raw, list):
            iterable = raw
        else:
            iterable = []
        for entry in iterable[:limit]:
            if isinstance(entry, dict):
                memory_text = entry.get("memory") or entry.get("content") or ""
                if memory_text:
                    items.append({"memory": memory_text, "score": entry.get("score"), "metadata": entry.get("metadata") or {}})
            elif entry:
                items.append({"memory": str(entry), "score": None, "metadata": {}})
        return items


memory_store = CustomerMemoryStore(settings=settings, llm=None)
print("Memory store ready.")

Memory store ready.


### Debug: Diagnose the 401 AuthenticationError

The Mem0 config in `mem0_store.py` sets `provider: "groq"` but passes `openai_api_key` / `openai_model`.  
Below we test each key independently to find the mismatch.

In [10]:
# ── 1. Inspect what Mem0 is actually configured with ──────────────────────────
print("=== Mem0 LLM config (from mem0_store.py) ===")
print(f"  provider       : groq          (hardcoded in mem0_store.py)")
print(f"  model          : {settings.openai_model}  (from settings.openai_model)")
print(f"  api_key (last4) : ...{settings.openai_api_key[-4:] if settings.openai_api_key else 'EMPTY'}")
print(f"  groq_api_key   : {'SET' if settings.groq_api_key else 'EMPTY'}")
print()

# ── 2. Test the Groq key directly ────────────────────────────────────────────
print("=== Test Groq API key ===")
try:
    import httpx
    resp = httpx.get(
        "https://api.groq.com/openai/v1/models",
        headers={"Authorization": f"Bearer {settings.groq_api_key}"},
        timeout=10,
    )
    if resp.status_code == 200:
        models = [m["id"] for m in resp.json().get("data", [])[:5]]
        print(f"  ✅ Groq key is VALID  (sample models: {models})")
    else:
        print(f"  ❌ Groq key returned {resp.status_code}: {resp.text[:200]}")
except Exception as exc:
    print(f"  ❌ Groq test failed: {exc}")

# ── 3. Test the OpenAI key directly ──────────────────────────────────────────
print("\n=== Test OpenAI API key ===")
try:
    resp = httpx.get(
        "https://api.openai.com/v1/models",
        headers={"Authorization": f"Bearer {settings.openai_api_key}"},
        timeout=10,
    )
    if resp.status_code == 200:
        models = [m["id"] for m in resp.json().get("data", [])[:5]]
        print(f"  ✅ OpenAI key is VALID  (sample models: {models})")
    else:
        print(f"  ❌ OpenAI key returned {resp.status_code}: {resp.text[:200]}")
except Exception as exc:
    print(f"  ❌ OpenAI test failed: {exc}")

# ── 4. Test the Google/Gemini key ────────────────────────────────────────────
print("\n=== Test Google API key (used for embeddings) ===")
try:
    resp = httpx.get(
        f"https://generativelanguage.googleapis.com/v1beta/models?key={settings.google_api_key}",
        timeout=10,
    )
    if resp.status_code == 200:
        print(f"  ✅ Google key is VALID")
    else:
        print(f"  ❌ Google key returned {resp.status_code}: {resp.text[:200]}")
except Exception as exc:
    print(f"  ❌ Google test failed: {exc}")

# ── 5. Root cause summary ────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("ROOT CAUSE ANALYSIS")
print("=" * 60)
print(
    "mem0_store.py line 22 sets  provider='groq'  but passes\n"
    "  settings.openai_api_key  and  settings.openai_model.\n"
    "\n"
    "FIX OPTIONS:\n"
    "  A) Change provider to 'openai' (if you want to use OpenAI):\n"
    '       "provider": "openai"\n'
    "  B) Use the Groq key + model (if you want to use Groq):\n"
    '       "model": settings.groq_model,\n'
    '       "api_key": settings.groq_api_key,\n'
)

=== Mem0 LLM config (from mem0_store.py) ===
  provider       : groq          (hardcoded in mem0_store.py)
  model          : gpt-4o  (from settings.openai_model)
  api_key (last4) : ...U1sA
  groq_api_key   : EMPTY

=== Test Groq API key ===
  ❌ Groq test failed: Illegal header value b'Bearer '

=== Test OpenAI API key ===
  ✅ OpenAI key is VALID  (sample models: ['gpt-4-0613', 'gpt-4', 'gpt-3.5-turbo', 'gpt-5.4-mini', 'gpt-5.4'])

=== Test Google API key (used for embeddings) ===
  ✅ Google key is VALID

ROOT CAUSE ANALYSIS
mem0_store.py line 22 sets  provider='groq'  but passes
  settings.openai_api_key  and  settings.openai_model.

FIX OPTIONS:
  A) Change provider to 'openai' (if you want to use OpenAI):
       "provider": "openai"
  B) Use the Groq key + model (if you want to use Groq):
       "model": settings.groq_model,
       "api_key": settings.groq_api_key,



In [11]:
# Add a sample interaction to memory
memory_store.add_interaction(
    user_id=customer["email"],
    user_input="I forgot my internet banking password. How do I reset it?",
    assistant_response=(
        "You can reset your password via the 'Forgot Password' link on the login page. "
        "You'll receive an OTP on your registered mobile number."
    ),
)
print("Interaction added to memory.")

Interaction added to memory.


In [12]:
# Search customer memory
mem_hits = memory_store.search(
    query="ATM withdrawal issue",
    user_id=customer["email"],
    limit=5,
)
print(f"Memory hits ({len(mem_hits)}):")
for hit in mem_hits:
    print(f"  - {hit['memory']}  (score: {hit.get('score')})")

Memory hits (0):


In [13]:
# List all memories for this customer
all_memories = memory_store.list_memories(user_id=customer["email"], limit=10)
print(f"All memories ({len(all_memories)}):")
for m in all_memories:
    print(f"  - {m['memory']}")

All memories (0):


## 5. Support Tools

LangChain `@tool`-decorated functions that the agent can call:  
- `lookup_customer_plan` — deterministic plan/SLA lookup  
- `lookup_open_ticket_load` — counts open tickets for the customer

In [ ]:
from langchain_core.tools import tool


def _stable_bucket(email: str, size: int) -> int:
    digest = hashlib.sha256(email.strip().lower().encode("utf-8")).hexdigest()
    return int(digest, 16) % size


def _json(payload: dict[str, Any]) -> str:
    return json.dumps(payload)


def _load_band(open_count: int) -> str:
    if open_count <= 1:
        return "light"
    if open_count <= 3:
        return "moderate"
    return "heavy"


@tool
def lookup_customer_plan(customer_email: str) -> str:
    """Return structured subscription and SLA details for a customer email."""
    plans = [
        {"plan_tier": "free", "sla_hours": 48, "priority_queue": False},
        {"plan_tier": "starter", "sla_hours": 24, "priority_queue": False},
        {"plan_tier": "pro", "sla_hours": 8, "priority_queue": True},
        {"plan_tier": "enterprise", "sla_hours": 1, "priority_queue": True},
    ]
    plan = plans[_stable_bucket(customer_email, len(plans))]
    summary = f"{customer_email} is on the {plan['plan_tier']} plan with {plan['sla_hours']}h SLA."
    return _json({
        "tool": "lookup_customer_plan",
        "customer_email": customer_email,
        "summary": summary,
        "details": plan,
        "recommended_action": "Use priority handling." if plan["priority_queue"] else "Use standard handling.",
    })


@tool
def lookup_open_ticket_load(customer_email: str) -> str:
    """Return open ticket count and load band for a customer email."""
    cust = customers_repo.get_by_email(customer_email)
    if not cust:
        return _json({
            "tool": "lookup_open_ticket_load",
            "customer_email": customer_email,
            "summary": f"No customer record found for {customer_email}.",
            "details": {"customer_found": False, "open_tickets": None, "load_band": "unknown"},
            "recommended_action": "Ask agent to verify customer email before promising SLA.",
        })
    open_count = tickets_repo.count_open_for_customer(customer_email)
    return _json({
        "tool": "lookup_open_ticket_load",
        "customer_email": customer_email,
        "summary": f"Customer {customer_email} has {open_count} open ticket(s).",
        "details": {"customer_found": True, "open_tickets": open_count, "load_band": _load_band(open_count)},
        "recommended_action": "Acknowledge multiple ongoing issues." if open_count > 1 else "Handle as isolated incident.",
    })


def get_support_tools() -> list:
    return [lookup_customer_plan, lookup_open_ticket_load]


# Invoke tools directly
plan_info = lookup_customer_plan.invoke({"customer_email": customer["email"]})
print("Plan info:", json.dumps(json.loads(plan_info), indent=2))

load_info = lookup_open_ticket_load.invoke({"customer_email": customer["email"]})
print("\nTicket load:", json.dumps(json.loads(load_info), indent=2))

Plan info: {
  "tool": "lookup_customer_plan",
  "customer_email": "alex@acme.io",
  "summary": "alex@acme.io is on the free plan with 48h SLA.",
  "details": {
    "plan_tier": "free",
    "sla_hours": 48,
    "priority_queue": false
  },
  "recommended_action": "Use standard handling."
}

Ticket load: {
  "tool": "lookup_open_ticket_load",
  "customer_email": "alex@acme.io",
  "summary": "No customer record found for alex@acme.io.",
  "details": {
    "customer_found": false,
    "open_tickets": null,
    "load_band": "unknown"
  },
  "recommended_action": "Ask agent to verify customer email before promising SLA."
}


In [ ]:
# List all available tools
tools = get_support_tools()
for t in tools:
    print(f"  Tool: {t.name} — {t.description}")

  Tool: lookup_customer_plan — Return structured subscription and SLA details for a customer email.
  Tool: lookup_open_ticket_load — Return open ticket count and load band for a customer email.


## 6. Copilot Agent — Draft Generation

`SupportCopilot` wires the LLM, tools, memory, and RAG together using a LangChain agent.  
Given a ticket + customer, it generates an empathetic draft reply.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver


class SupportCopilot:
    def __init__(self, settings: Settings):
        if not settings.openai_api_key:
            raise RuntimeError("OPENAI_API_KEY is missing. Add it in .env before generating drafts.")
        self._settings = settings
        self._llm = ChatOpenAI(
            model=settings.openai_model,
            api_key=settings.openai_api_key,
            temperature=settings.llm_temperature,
        )
        self._tools = get_support_tools()
        self._agent = create_agent(
            model=self._llm,
            tools=self._tools,
            checkpointer=InMemorySaver(),
            name="support_copilot_agent",
        )
        self._memory_error: str | None = None
        try:
            self.memory = CustomerMemoryStore(settings=settings, llm=self._llm)
        except Exception as exc:
            self._memory_error = str(exc)
        self.rag = KnowledgeBaseService(settings=settings)

    def generate_draft(self, ticket: dict[str, Any], customer: dict[str, Any]) -> dict[str, Any]:
        query = f"{ticket['subject']}\n{ticket['description']}"
        customer_email = customer["email"]
        memory_hits = self._search_memory_scopes(
            query=query, customer_email=customer_email,
            customer_company=customer.get("company"), limit=self._settings.mem0_top_k,
        )
        kb_hits = self.rag.search(query=query, top_k=self._settings.rag_top_k)
        system_prompt = self._build_system_prompt(memory_hits=memory_hits, kb_hits=kb_hits)
        user_prompt = self._build_user_prompt(ticket=ticket, customer=customer)
        agent_result = self._agent.invoke(
            {"messages": [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]},
            config={
                "configurable": {"thread_id": self._thread_id_for_ticket(ticket=ticket, customer=customer)},
                "recursion_limit": 40,
            },
        )
        draft_text, tool_calls = self._extract_agent_draft_and_tool_calls(agent_result)
        used_fallback = False
        if not draft_text:
            draft_text = self._fallback_generate_text(
                ticket=ticket, customer=customer, memory_hits=memory_hits, kb_hits=kb_hits, tool_calls=tool_calls,
            )
            used_fallback = True
        if not draft_text:
            draft_text = self._deterministic_fallback(ticket=ticket, customer=customer, tool_calls=tool_calls)
            used_fallback = True
        context_used = self._build_context(
            ticket=ticket, customer=customer, memory_hits=memory_hits, kb_hits=kb_hits, tool_calls=tool_calls,
        )
        if self._memory_error:
            context_used.setdefault("errors", []).append(f"Memory disabled: {self._memory_error}")
        if used_fallback:
            context_used.setdefault("errors", []).append(
                "Primary tool-call response had empty content; fallback synthesis was used."
            )
        context_used["agent_runtime"] = "langchain_create_agent"
        return {"draft": draft_text, "context_used": context_used}

    def save_accepted_resolution(self, customer_email: str, customer_company: str | None,
                                  ticket_subject: str, ticket_description: str,
                                  draft_content: str, context_used: dict[str, Any] | None = None) -> None:
        entity_links = self._extract_entity_links(
            ticket_subject=ticket_subject, ticket_description=ticket_description,
            draft_content=draft_content, context_used=context_used or {},
        )
        for scope_user_id in self._memory_scope_ids(customer_email=customer_email, customer_company=customer_company):
            self.memory.add_resolution(
                user_id=scope_user_id, ticket_subject=ticket_subject,
                ticket_description=ticket_description, accepted_draft=draft_content, entity_links=entity_links,
            )

    def list_customer_memories(self, customer_email: str, customer_company: str | None = None, limit: int = 20) -> list[dict[str, Any]]:
        scope_user_ids = self._memory_scope_ids(customer_email=customer_email, customer_company=customer_company)
        raw_hits: list[dict[str, Any]] = []
        for scope_user_id in scope_user_ids:
            hits = self.memory.list_memories(user_id=scope_user_id, limit=max(1, limit))
            raw_hits.extend(self._annotate_memory_scope(hits=hits, scope_user_id=scope_user_id))
        return self._dedupe_memory_hits(raw_hits, limit=max(1, limit))

    def search_customer_memories(self, customer_email: str, query: str, customer_company: str | None = None, limit: int = 10) -> list[dict[str, Any]]:
        return self._search_memory_scopes(query=query, customer_email=customer_email, customer_company=customer_company, limit=limit)

    def _search_memory_scopes(self, query: str, customer_email: str, customer_company: str | None, limit: int) -> list[dict[str, Any]]:
        per_scope_limit = max(1, limit)
        scope_user_ids = self._memory_scope_ids(customer_email=customer_email, customer_company=customer_company)
        raw_hits: list[dict[str, Any]] = []
        for scope_user_id in scope_user_ids:
            hits = self.memory.search(query=query, user_id=scope_user_id, limit=per_scope_limit)
            raw_hits.extend(self._annotate_memory_scope(hits=hits, scope_user_id=scope_user_id))
        return self._dedupe_memory_hits(raw_hits, limit=per_scope_limit * len(scope_user_ids))

    def _memory_scope_ids(self, customer_email: str, customer_company: str | None) -> list[str]:
        scope_user_ids = [customer_email.strip().lower()]
        company_scope = self._company_scope_user_id(customer_company)
        if company_scope:
            scope_user_ids.append(company_scope)
        return self._unique_ordered(scope_user_ids)

    @staticmethod
    def _company_scope_user_id(customer_company: str | None) -> str | None:
        if not customer_company:
            return None
        lowered = customer_company.strip().lower()
        if not lowered:
            return None
        normalized = re.sub(r"[^a-z0-9]+", "-", lowered).strip("-")
        return f"company::{normalized}" if normalized else None

    @staticmethod
    def _annotate_memory_scope(hits: list[dict[str, Any]], scope_user_id: str) -> list[dict[str, Any]]:
        scope = "company" if scope_user_id.startswith("company::") else "customer"
        annotated = []
        for hit in hits:
            item = dict(hit)
            metadata = dict(item.get("metadata") or {})
            metadata.setdefault("scope", scope)
            metadata.setdefault("scope_user_id", scope_user_id)
            item["metadata"] = metadata
            annotated.append(item)
        return annotated

    @staticmethod
    def _dedupe_memory_hits(hits: list[dict[str, Any]], limit: int) -> list[dict[str, Any]]:
        deduped, seen = [], set()
        for hit in hits:
            memory_text = str(hit.get("memory", "")).strip()
            if not memory_text:
                continue
            key = memory_text.lower()
            if key in seen:
                continue
            seen.add(key)
            deduped.append(hit)
            if len(deduped) >= max(1, limit):
                break
        return deduped

    @staticmethod
    def _extract_content(response: Any) -> str:
        content = getattr(response, "content", response)
        if isinstance(content, list):
            return "\n".join(str(item) for item in content)
        return str(content)

    @staticmethod
    def _format_memory(memory_hits: list[dict[str, Any]]) -> str:
        if not memory_hits:
            return "- No prior customer memories found."
        return "\n".join(f"- {item.get('memory', '').strip()}" for item in memory_hits)

    @staticmethod
    def _format_kb(kb_hits: list[dict[str, Any]]) -> str:
        if not kb_hits:
            return "- No relevant knowledge-base chunks found."
        return "\n".join(f"- [{item.get('source', 'unknown')}] {item.get('content', '').strip()}" for item in kb_hits)

    def _build_system_prompt(self, memory_hits: list[dict[str, Any]], kb_hits: list[dict[str, Any]]) -> str:
        return (
            "You are an AI copilot for customer support agents. "
            "Write concise, empathetic, and actionable draft replies. "
            "If needed, call tools to verify plan, billing, or ticket load before finalizing.\n\n"
            "Customer Memory Context:\n"
            f"{self._format_memory(memory_hits)}\n\n"
            "Knowledge Base Context:\n"
            f"{self._format_kb(kb_hits)}\n\n"
            "Output rules:\n"
            "1) Start with empathy and direct acknowledgement.\n"
            "2) Provide clear next steps or resolution path.\n"
            "3) Reference KB/tool facts when relevant, without exposing internal chain-of-thought.\n"
            "4) Keep response under 180 words unless more detail is necessary."
        )

    @staticmethod
    def _build_user_prompt(ticket: dict[str, Any], customer: dict[str, Any]) -> str:
        return (
            f"Customer: {customer.get('name') or 'Unknown'} ({customer['email']})\n"
            f"Company: {customer.get('company') or 'Unknown'}\n"
            f"Ticket Subject: {ticket['subject']}\n"
            f"Ticket Priority: {ticket.get('priority', 'medium')}\n"
            f"Ticket Description:\n{ticket['description']}\n\n"
            "Create a draft response for the support agent. "
            "Use tools when the ticket likely needs billing, plan, or account-level checks."
        )

    @staticmethod
    def _thread_id_for_ticket(ticket: dict[str, Any], customer: dict[str, Any]) -> str:
        ticket_id = ticket.get("id")
        if ticket_id is not None:
            return f"ticket::{ticket_id}"
        customer_email = str(customer.get("email") or "").strip().lower()
        return f"ticket::{customer_email}" if customer_email else "ticket::unknown"

    def _extract_agent_draft_and_tool_calls(self, agent_result: Any) -> tuple[str, list[dict[str, Any]]]:
        if isinstance(agent_result, dict):
            raw_messages = agent_result.get("messages") or []
        else:
            raw_messages = getattr(agent_result, "messages", []) or []
        messages = [item for item in raw_messages if isinstance(item, BaseMessage)]
        draft_text = ""
        for message in reversed(messages):
            if not isinstance(message, AIMessage):
                continue
            candidate = self._extract_content(message).strip()
            if candidate:
                draft_text = candidate
                break
        tool_messages_by_id: dict[str, ToolMessage] = {}
        for message in messages:
            if isinstance(message, ToolMessage) and message.tool_call_id:
                tool_messages_by_id[message.tool_call_id] = message
        tool_calls: list[dict[str, Any]] = []
        for message in messages:
            if not isinstance(message, AIMessage):
                continue
            pending_calls = getattr(message, "tool_calls", None) or []
            for call in pending_calls:
                tool_name = call.get("name")
                tool_id = call.get("id")
                args = call.get("args")
                safe_tool_name = tool_name or "unknown_tool"
                trace: dict[str, Any] = {
                    "tool_name": safe_tool_name, "tool_call_id": tool_id,
                    "arguments": args if isinstance(args, dict) else {},
                }
                tool_message = tool_messages_by_id.get(str(tool_id)) if tool_id is not None else None
                if not tool_message:
                    trace.update({
                        "status": "skipped",
                        "summary": f"Tool '{safe_tool_name}' was requested but no result was returned.",
                        "output": None, "output_text": f"Tool '{safe_tool_name}' produced no output.",
                    })
                    tool_calls.append(trace)
                    continue
                output_text = self._extract_content(tool_message)
                parsed_output, output_text = self._parse_tool_output(output_text)
                summary = self._tool_summary(parsed_output=parsed_output, output_text=output_text)
                status = "error" if getattr(tool_message, "status", None) == "error" else "ok"
                trace.update({"status": status, "summary": summary, "output": parsed_output, "output_text": output_text})
                tool_calls.append(trace)
        return draft_text, tool_calls

    @staticmethod
    def _parse_tool_output(raw_output: Any) -> tuple[dict[str, Any] | None, str]:
        if isinstance(raw_output, dict):
            return raw_output, json.dumps(raw_output)
        output_text = str(raw_output)
        try:
            parsed = json.loads(output_text)
            if isinstance(parsed, dict):
                return parsed, output_text
        except json.JSONDecodeError:
            pass
        return None, output_text

    @staticmethod
    def _tool_summary(parsed_output: dict[str, Any] | None, output_text: str) -> str:
        if parsed_output:
            summary = parsed_output.get("summary")
            if summary:
                return str(summary)
        return output_text

    def _build_context(self, ticket: dict[str, Any], customer: dict[str, Any],
                       memory_hits: list[dict[str, Any]], kb_hits: list[dict[str, Any]],
                       tool_calls: list[dict[str, Any]]) -> dict[str, Any]:
        knowledge_sources = self._unique_ordered(
            [str(item.get("source")) for item in kb_hits if item.get("source")]
        )
        tool_errors = [item for item in tool_calls if item.get("status") != "ok"]
        return {
            "version": 2,
            "ticket": {"id": ticket.get("id"), "subject": ticket.get("subject"),
                       "priority": ticket.get("priority"), "status": ticket.get("status")},
            "customer": {"id": customer.get("id"), "email": customer.get("email"),
                         "name": customer.get("name"), "company": customer.get("company")},
            "signals": {
                "memory_hit_count": len(memory_hits), "knowledge_hit_count": len(kb_hits),
                "tool_call_count": len(tool_calls), "tool_error_count": len(tool_errors),
                "knowledge_sources": knowledge_sources,
            },
            "highlights": {
                "memory": [self._trim_text(item.get("memory", "")) for item in memory_hits[:3]],
                "knowledge": [self._trim_text(f"[{item.get('source', 'unknown')}] {item.get('content', '')}") for item in kb_hits[:3]],
                "tools": [self._trim_text(item.get("summary", "")) for item in tool_calls[:3]],
            },
            "memory_hits": memory_hits, "knowledge_hits": kb_hits, "tool_calls": tool_calls,
        }

    @staticmethod
    def _unique_ordered(values: list[str]) -> list[str]:
        seen, ordered = set(), []
        for value in values:
            if value in seen:
                continue
            seen.add(value)
            ordered.append(value)
        return ordered

    @staticmethod
    def _trim_text(text: Any, limit: int = 180) -> str:
        clean = str(text or "").strip()
        return clean if len(clean) <= limit else f"{clean[:limit - 3]}..."

    def _extract_entity_links(self, ticket_subject: str, ticket_description: str,
                               draft_content: str, context_used: dict[str, Any]) -> list[str]:
        merged_text = f"{ticket_subject}\n{ticket_description}\n{draft_content}"
        merged_lower = merged_text.lower()
        links: list[str] = []
        endpoints = re.findall(r"/[a-zA-Z0-9][a-zA-Z0-9/_-]{2,}", merged_text)
        for endpoint in self._unique_ordered(endpoints)[:3]:
            links.append(f"endpoint:{endpoint}")
        status_codes = re.findall(r"\b([45]\d\d)\b", merged_text)
        for code in self._unique_ordered(status_codes)[:4]:
            links.append(f"http_status:{code}")
        regions = [("EU", [" eu ", "europe", "emea"]), ("US", [" us ", "united states", "na "]),
                    ("APAC", [" apac ", "asia pacific"]), ("India", [" india ", " in "])]
        padded = f" {merged_lower} "
        for region, markers in regions:
            if any(marker in padded for marker in markers):
                links.append(f"region:{region}")
        integrations = ["shopify", "stripe", "salesforce", "slack", "quickbooks", "hubspot", "zendesk"]
        for integration in integrations:
            if integration in merged_lower:
                links.append(f"integration:{integration}")
        for tool_call in context_used.get("tool_calls", []):
            output = tool_call.get("output") or {}
            details = output.get("details") if isinstance(output, dict) else None
            if not isinstance(details, dict):
                continue
            plan = details.get("plan_tier")
            if plan:
                links.append(f"plan:{plan}")
            risk = details.get("risk_level")
            if risk:
                links.append(f"billing_risk:{risk}")
        return self._unique_ordered([item for item in links if item])[:12]

    def _fallback_generate_text(self, ticket: dict[str, Any], customer: dict[str, Any],
                                 memory_hits: list[dict[str, Any]], kb_hits: list[dict[str, Any]],
                                 tool_calls: list[dict[str, Any]]) -> str:
        tool_summaries = [self._trim_text(item.get("summary") or item.get("output_text", "")) for item in tool_calls if item.get("summary") or item.get("output_text")]
        memory_summaries = [self._trim_text(item.get("memory", "")) for item in memory_hits[:3]]
        kb_summaries = [self._trim_text(f"[{item.get('source', 'unknown')}] {item.get('content', '')}") for item in kb_hits[:3]]
        fallback_system = "You are an AI support copilot. Produce only the final customer-facing draft reply. No tool calls."
        fallback_user = (
            f"Customer: {customer.get('name') or 'Unknown'} ({customer.get('email', 'unknown')})\n"
            f"Company: {customer.get('company') or 'Unknown'}\n"
            f"Ticket subject: {ticket.get('subject', '')}\n"
            f"Ticket description: {ticket.get('description', '')}\n\n"
            "Memory highlights:\n"
            f"{chr(10).join('- ' + item for item in memory_summaries) if memory_summaries else '- none'}\n\n"
            "Knowledge highlights:\n"
            f"{chr(10).join('- ' + item for item in kb_summaries) if kb_summaries else '- none'}\n\n"
            "Tool findings:\n"
            f"{chr(10).join('- ' + item for item in tool_summaries) if tool_summaries else '- none'}\n\n"
            "Write a concise, empathetic draft with clear next steps."
        )
        try:
            response = self._llm.invoke([SystemMessage(content=fallback_system), HumanMessage(content=fallback_user)])
            return self._extract_content(response).strip()
        except Exception:
            return ""

    def _deterministic_fallback(self, ticket: dict[str, Any], customer: dict[str, Any], tool_calls: list[dict[str, Any]]) -> str:
        customer_name = customer.get("name") or customer.get("email") or "there"
        best_tool_summary = ""
        for item in tool_calls:
            summary = str(item.get("summary") or "").strip()
            if summary:
                best_tool_summary = summary
                break
        action_line = best_tool_summary if best_tool_summary else "Our support team is reviewing your account and issue details now."
        return (
            f"Hi {customer_name},\n\n"
            f"Thanks for reaching out about \"{ticket.get('subject', 'your issue')}\". "
            "I understand how disruptive this can be.\n\n"
            f"{action_line}\n\n"
            "Next, we will continue investigating and share an update with concrete steps shortly.\n\n"
            "Best,\nSupport Team"
        )


copilot = SupportCopilot(settings=settings)
print("Copilot ready.")
print(f"  LLM model   : {settings.openai_model}")
print(f"  Tools       : {[t.name for t in copilot._tools]}")
print(f"  Memory OK   : {copilot._memory_error is None}")

Copilot ready.
  LLM model   : gpt-4o
  Tools       : ['lookup_customer_plan', 'lookup_open_ticket_load']
  Memory OK   : True


In [17]:
# Enrich the ticket dict with customer fields (as the agent expects)
ticket_for_copilot = tickets_repo.get_by_id(ticket["id"])
customer_for_copilot = customers_repo.get_by_id(customer["id"])

result = copilot.generate_draft(
    ticket=ticket_for_copilot,
    customer=customer_for_copilot,
)

draft_text = result["draft"]
context_used = result["context_used"]

print("=" * 60)
print("GENERATED DRAFT")
print("=" * 60)
print(draft_text)
print("\n" + "=" * 60)
print("CONTEXT — signals")
print("=" * 60)
print(json.dumps(context_used.get("signals", {}), indent=2))

GENERATED DRAFT
Hi Alex,

I'm sorry to hear about the trouble with your ATM withdrawal. I understand how important it is to have access to your funds. Typically, if cash is debited but not dispensed, the reversal is processed within 24 hours. Since this hasn't happened, I'll guide you on the next steps.

Please provide the following details to expedite the dispute process:
- Transaction date and time
- ATM location
- Last 4 digits of your card

Once we have this information, we can initiate a formal dispute to ensure the amount is reversed promptly.

Additionally, it seems there might be a discrepancy with your account details in our system. Could you please confirm your registered email address with us? This will help us ensure timely support.

Thank you for your patience and understanding.

Best regards,
[Your Name]

CONTEXT — signals
{
  "memory_hit_count": 0,
  "knowledge_hit_count": 4,
  "tool_call_count": 2,
  "tool_error_count": 0,
  "knowledge_sources": [
    "banking-atm-cash-

In [ ]:
# Inspect tool calls made by the agent
for tc in context_used.get("tool_calls", []):
    print(f"Tool: {tc['tool_name']}  Status: {tc['status']}")
    print(f"  Summary: {tc.get('summary', '')}")
    print()

## 7. Draft Service — End-to-End Workflow

`DraftService` orchestrates ticket lookup, copilot invocation, and draft storage — the same logic the API uses, but called directly.

In [ ]:
class DraftService:
    def serialize_draft(self, draft: dict[str, Any]) -> dict[str, Any]:
        context_raw = draft.get("context_used")
        context_data: dict[str, Any] | None = None
        if context_raw:
            try:
                context_data = json.loads(context_raw)
            except json.JSONDecodeError:
                context_data = {"raw": context_raw}
        return {
            "id": draft["id"], "ticket_id": draft["ticket_id"], "content": draft["content"],
            "context_used": context_data, "status": draft["status"], "created_at": draft["created_at"],
        }

    def serialize_ticket(self, ticket: dict[str, Any]) -> dict[str, Any]:
        return {
            "id": ticket["id"], "customer_id": ticket["customer_id"],
            "customer_email": ticket["customer_email"], "customer_name": ticket.get("customer_name"),
            "customer_company": ticket.get("customer_company"), "subject": ticket["subject"],
            "description": ticket["description"], "status": ticket["status"],
            "priority": ticket["priority"], "created_at": ticket["created_at"], "updated_at": ticket["updated_at"],
        }

    def parse_context_used(self, raw: Any) -> dict[str, Any]:
        if isinstance(raw, dict):
            return raw
        if isinstance(raw, str) and raw:
            try:
                parsed = json.loads(raw)
                return parsed if isinstance(parsed, dict) else {"raw": raw}
            except json.JSONDecodeError:
                return {"raw": raw}
        return {}

    def generate_and_store_background(self, ticket_id: int, tickets_repo: TicketsRepository,
                                       customers_repo: CustomersRepository, drafts_repo: DraftsRepository,
                                       copilot_factory: Callable[[], SupportCopilot], logger: logging.Logger) -> dict[str, Any] | None:
        ticket = tickets_repo.get_by_id(ticket_id)
        if not ticket:
            return None
        customer = customers_repo.get_by_id(ticket["customer_id"])
        if not customer:
            return None
        try:
            cp = copilot_factory()
            result = cp.generate_draft(ticket=ticket, customer=customer)
            draft_text, context_used = self._normalize_draft_result(result)
            return drafts_repo.create(ticket_id=ticket_id, content=draft_text,
                                      context_used=json.dumps(context_used), status="pending")
        except Exception as exc:
            logger.exception("Background draft generation failed for ticket_id=%s", ticket_id)
            return drafts_repo.create(
                ticket_id=ticket_id,
                content="Automatic draft generation failed. Configure AI keys and trigger manual draft generation.",
                context_used=json.dumps(self._failed_context(str(exc))), status="failed",
            )

    def generate_and_store_manual(self, ticket_id: int, ticket: dict[str, Any], customer: dict[str, Any],
                                   drafts_repo: DraftsRepository, copilot: SupportCopilot) -> dict[str, Any]:
        result = copilot.generate_draft(ticket=ticket, customer=customer)
        draft_text, context_used = self._normalize_draft_result(result)
        return drafts_repo.create(ticket_id=ticket_id, content=draft_text,
                                  context_used=json.dumps(context_used), status="pending")

    def _normalize_draft_result(self, result: dict[str, Any]) -> tuple[str, dict[str, Any]]:
        draft_text = str(result.get("draft") or "").strip()
        context = result.get("context_used") or {}
        if not isinstance(context, dict):
            context = {"raw": str(context)}
        if not draft_text:
            draft_text = "Thanks for your message. We are reviewing your issue and will share a concrete update shortly."
            context.setdefault("errors", []).append("Copilot returned empty draft content; API fallback text was used.")
        return draft_text, context

    @staticmethod
    def _failed_context(error_text: str) -> dict[str, Any]:
        return {
            "version": 2,
            "signals": {"memory_hit_count": 0, "knowledge_hit_count": 0,
                        "tool_call_count": 0, "tool_error_count": 1, "knowledge_sources": []},
            "highlights": {"memory": [], "knowledge": [], "tools": []},
            "memory_hits": [], "knowledge_hits": [], "tool_calls": [], "errors": [error_text],
        }


draft_service = DraftService()

stored_draft = draft_service.generate_and_store_manual(
    ticket_id=ticket["id"],
    ticket=ticket_for_copilot,
    customer=customer_for_copilot,
    drafts_repo=drafts_repo,
    copilot=copilot,
)

print("Stored draft:", json.dumps(draft_service.serialize_draft(stored_draft), indent=2, default=str))

Stored draft: {
  "id": 1,
  "ticket_id": 5,
  "content": "Hi Alex,\n\nI'm sorry to hear about the issue with your ATM withdrawal. I understand the urgency of having your funds available. When cash is debited but not dispensed, the reversal usually occurs within 24 hours. Since this hasn't happened, let's proceed with a dispute.\n\nPlease provide:\n- Transaction date and time\n- ATM location\n- Last 4 digits of your card\n\nThis information will help us expedite the reversal process.\n\nAdditionally, it seems there might be a discrepancy with your account details in our system. Could you please confirm your registered email address with us? This will ensure we provide timely support.\n\nThank you for your patience.\n\nBest regards,\n[Your Name]",
  "context_used": {
    "version": 2,
    "ticket": {
      "id": 5,
      "subject": "Unable to withdraw cash from ATM",
      "priority": "high",
      "status": "open"
    },
    "customer": {
      "id": 1,
      "email": "alex@acme.io",
 

In [19]:
# Simulate accepting the draft (saves resolution to memory)
drafts_repo.update(draft_id=stored_draft["id"], status="accepted")

copilot.save_accepted_resolution(
    customer_email=customer["email"],
    customer_company=customer.get("company"),
    ticket_subject=ticket_for_copilot["subject"],
    ticket_description=ticket_for_copilot["description"],
    draft_content=stored_draft["content"],
    context_used=draft_service.parse_context_used(stored_draft.get("context_used")),
)

print("Resolution saved to customer memory.")

Resolution saved to customer memory.


In [20]:
# Verify resolution appears in memory
post_resolution_memories = memory_store.search(
    query="ATM withdrawal",
    user_id=customer["email"],
    limit=5,
)
print(f"Memories after resolution ({len(post_resolution_memories)}):")
for m in post_resolution_memories:
    print(f"  - {m['memory']}")

Memories after resolution (0):


## 8. Knowledge Service — Convenience Wrapper

In [ ]:
class KnowledgeService:
    def __init__(self, settings: Settings):
        self._settings = settings

    def ingest(self, clear_existing: bool = False) -> dict[str, int]:
        rag_service = KnowledgeBaseService(settings=self._settings)
        return rag_service.ingest_directory(
            directory=self._settings.knowledge_base_path,
            clear_existing=clear_existing,
        )


knowledge_svc = KnowledgeService(settings=settings)
ingest_stats = knowledge_svc.ingest(clear_existing=False)
print("Knowledge ingest stats:", json.dumps(ingest_stats, indent=2))

Knowledge ingest stats: {
  "files_indexed": 4,
  "chunks_indexed": 4,
  "collection_count": 4
}


---

**Summary**: This notebook exercised every layer of the `customer_support_agent` package — settings, DB repositories, RAG knowledge base, Mem0 memory, LangChain tools, the copilot agent, and the draft service — all without any FastAPI server.